In [2]:
import torch
import torchvision

import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import os

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)
print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
weights = models.ResNet18_Weights.DEFAULT

transform = weights.transforms()

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print(len(train_dataset))
print(len(test_dataset))

50000
10000


In [5]:
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

print(len(train_dataset))
print(len(val_dataset))

40000
10000


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [7]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    10
)

model = model.to(device)

In [9]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable Parameters: {total_params:,}")

Trainable Parameters: 11,181,642


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [14]:
def train_model(model, train_loader, val_loader,
                criterion, optimizer, epochs):

    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):

        # Training phase
        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Validation phase
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)

                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Loss: {avg_loss:.4f} | "
            f"Val Accuracy: {val_acc:.2f}%"
        )

    return train_losses, val_accuracies



In [16]:
resnet_losses, resnet_val_acc = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=10
)

Epoch [1/10] | Loss: 0.5338 | Val Accuracy: 85.77%
Epoch [2/10] | Loss: 0.2992 | Val Accuracy: 86.54%
Epoch [3/10] | Loss: 0.1906 | Val Accuracy: 87.24%
Epoch [4/10] | Loss: 0.1392 | Val Accuracy: 89.28%
Epoch [5/10] | Loss: 0.1053 | Val Accuracy: 88.26%
Epoch [6/10] | Loss: 0.0765 | Val Accuracy: 88.28%
Epoch [7/10] | Loss: 0.0653 | Val Accuracy: 89.72%
Epoch [8/10] | Loss: 0.0624 | Val Accuracy: 89.05%
Epoch [9/10] | Loss: 0.0548 | Val Accuracy: 88.50%
Epoch [10/10] | Loss: 0.0418 | Val Accuracy: 89.17%


In [17]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

resnet_test_acc = 100 * correct / total

print(f"ResNet18 Test Accuracy: {resnet_test_acc:.2f}%")

ResNet18 Test Accuracy: 88.60%
